# coder-1 — Colab eğitim turu
Kaggle'a paralel ikinci hat. Runtime → Change runtime type → **T4 GPU** seç, sonra **Runtime → Run all**.
Cell 1 sende kaggle.json yoksa `~/.kaggle/kaggle.json` içeriğini yapıştırma yeri gösterir.
Bitince sonuç `/content/c1_out/runs/` altında; Cell 4 onu `c1-colab-out` dataset'ine yükler (ben oradan alıp siteye takarım).

In [ ]:
# 1) Kaggle auth + araçlar
import os, getpass, json, pathlib
os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('kaggle.json içeriğini yapıştır (Kaggle → Settings → Create New Token):')
    txt = getpass.getpass('kaggle.json> ')
    pathlib.Path('/root/.kaggle/kaggle.json').write_text(txt)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip -q install kaggle
print('kaggle hazır')

In [ ]:
# 2) Kodu aç + datasetleri indir
import base64, io, tarfile, os
ROOT = '/content/coder-1'
os.makedirs(ROOT, exist_ok=True)
_SRC_B64 = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(_SRC_B64)), mode='r:gz') as t:
    t.extractall(ROOT)
os.chdir(ROOT)
import sys; sys.path.insert(0, ROOT)
!pip -q install -r requirements.txt
!kaggle datasets download chartescode/c1-round1 -p /content/r1 --unzip -q
!kaggle datasets download chartescode/c1-data-v2 -p /content/v2 --unzip -q
print('hazır:', os.listdir('/content/v2'))

In [ ]:
# 3) Eğitim: Tur-1 ckpt'ten devam pretrain + SFT
import subprocess, torch, os
assert torch.cuda.is_available(), 'GPU yok — Runtime tipini T4 yap!'
def sh(c):
    print('+', c, flush=True); subprocess.run(c, shell=True, check=True, cwd='/content/coder-1')
sh('python -m model.pretrain --preset mini --data /content/v2/shards --tokenizer /content/v2/tok16k '
   '--out /content/c1_out/runs/mini_v2 --steps 6000 --batch 8 --accum 8 --seq 512 '
   '--lr 2e-4 --ckpt_every 500 --sample_every 2000 --device cuda '
   '--init_from /content/r1/mini/last')
sh('python -m model.sft --ckpt /content/c1_out/runs/mini_v2/last '
   '--data /content/r1/sft.jsonl --tokenizer /content/v2/tok16k '
   '--out /content/c1_out/runs/sft --steps 800 --batch 4 --accum 4 --seq 512 --lr 1e-4 --device cuda')
sh('python -m model.eval --ckpt /content/c1_out/runs/sft/last '
   '--tokenizer /content/v2/tok16k --out /content/c1_out/eval_sft.json --device cuda')
print('TUR BİTTİ')

In [ ]:
# 4) Çıktıyı dataset'e yükle (ben oradan siteye takarım)
import os, json, subprocess
meta = {'id':'chartescode/c1-colab-out','title':'coder-1 colab output','is_private':True,'licenses':[{'name':'other'}]}
json.dump(meta, open('/content/c1_out/dataset-metadata.json','w'))
r = subprocess.run('kaggle datasets list -m | grep c1-colab-out', shell=True, capture_output=True, text=True)
if 'c1-colab-out' in r.stdout:
    os.system('kaggle datasets version -p /content/c1_out -m "tur çıktısı" --dir-mode zip')
else:
    os.system('kaggle datasets create -p /content/c1_out --dir-mode zip')
print('yüklendi — Devin alacak')